# ATP Matches Cleaning Pipeline (2016–2025)

This notebook ingests the per-year ATP match CSVs (2016 through 2025) plus an enriched player database, and produces a single analysis-ready dataset. It handles encoding issues (Cyrillic, accented Latin, mojibake), drops invalid matches (walkovers / retirements / Davis Cup ties), remaps Carpet to Hard, imputes missing serve stats, and writes a canonical `atp_matches_clean_final.parquet` used by the downstream PySpark feature-engineering notebook.

**Why 2016+:** Pre-2016 data has ~12% imputed serve stats (vs ~4% post-2016) due to gaps in Jeff Sackmann's collection — synthetic stats in 1-in-8 matches distort rolling-average features at training time.

**Why no Davis Cup:** ~63% of DC rows have imputed serve stats (home venues, non-standard surfaces), making them unreliable for tour-level modelling.

**Environment:** Databricks Serverless. 2016–2024 match files are pulled directly from Jeff Sackmann's GitHub mirror; 2025 is read from the user's Workspace folder (freshly scraped from SofaScore, not yet in the GitHub mirror).

## Section 1 — Setup & Load

Load each `atp_matches_YYYY.csv` for 2016–2024 directly from GitHub, load 2025 from the local Workspace path, tag each row with its source year, and concatenate into a single DataFrame `df_raw`. Also load the enriched player database into `df_players` for later lookup.

In [ ]:
import os
import unicodedata

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

In [ ]:
GITHUB_URL_TEMPLATE = 'https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_{year}.csv'
LOCAL_2025_PATH     = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/datasets/atp_matches_2025.csv'
PLAYERS_PATH        = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/datasets/atp_players.csv'
OUTPUT_DIR          = '/Workspace/Users/f.chiesa28@ncf.edu/Octenpus/datasets'

frames = []
col_signatures = set()

for year in range(2016, 2025):
    url = GITHUB_URL_TEMPLATE.format(year=year)
    d = pd.read_csv(url, low_memory=False)
    d['year'] = year
    col_signatures.add(tuple(c for c in d.columns if c != 'year'))
    frames.append(d)
    print(f'  loaded {year} from GitHub  shape={d.shape}')

d_2025 = pd.read_csv(LOCAL_2025_PATH, low_memory=False)
d_2025['year'] = 2025
col_signatures.add(tuple(c for c in d_2025.columns if c != 'year'))
frames.append(d_2025)
print(f'  loaded 2025 from Workspace  shape={d_2025.shape}')

df_raw = pd.concat(frames, ignore_index=True)
print(f'\ndf_raw shape: {df_raw.shape}')
print(f'Distinct column signatures across years: {len(col_signatures)}  (1 means all years share the same columns)')

In [ ]:
df_players = pd.read_csv(PLAYERS_PATH, low_memory=False)
print(f'Loaded players file: {os.path.basename(PLAYERS_PATH)}  shape={df_players.shape}')
print('Columns:', list(df_players.columns))

## Section 2 — Drop unused columns

Keep only the columns required for downstream feature engineering. Everything else (seed, entry, draw size, rank points, minutes, player IDs, …) is dropped silently.

In [ ]:
KEEP_COLS = [
    'tourney_id', 'tourney_name', 'surface', 'tourney_level', 'tourney_date',
    'match_num', 'round', 'best_of', 'year',
    'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age',
    'loser_name',  'loser_hand',  'loser_ht',  'loser_ioc',  'loser_age',
    'score',
    'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
    'w_SvGms', 'w_bpSaved', 'w_bpFaced',
    'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon',
    'l_SvGms', 'l_bpSaved', 'l_bpFaced',
    'winner_rank', 'loser_rank',
]

print(f'Shape before drop: {df_raw.shape}')
df = df_raw[[c for c in KEEP_COLS if c in df_raw.columns]].copy()
print(f'Shape after drop:  {df.shape}')
print(f'\u2713 Section 2 complete \u2014 kept {df.shape[1]} columns')

## Section 3 — Character encoding normalization

Many player and tournament names contain non-ASCII characters (Cyrillic, Arabic, Chinese, accented Latin). During scraping / CSV export some of these were corrupted into *mojibake* (e.g. `Ã¡` where `á` was meant). We fix both composed-form issues (NFC) and mojibake before standardizing the `hand` column.

In [ ]:
MOJIBAKE_MAP = {
    'Ã¡': 'á', 'Ã©': 'é', 'Ã\xad': 'í', 'Ã³': 'ó', 'Ãº': 'ú',
    'Ã±': 'ñ', 'Ã ': 'à', 'Ã¨': 'è', 'Ã¬': 'ì', 'Ã²': 'ò',
    'Ä\x87': 'ć', 'Ä\x91': 'đ', 'Å¡': 'š', 'Å¾': 'ž', 'Ä\x8d': 'č',
    'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã¤': 'ä', 'ÃŸ': 'ß',
    'Ã\x87': 'Ç', 'Ã\x89': 'É', 'Ã\x93': 'Ó', 'Ãœ': 'Ü',
}

def normalize_text(s):
    if not isinstance(s, str):
        return s
    for bad, good in MOJIBAKE_MAP.items():
        if bad in s:
            s = s.replace(bad, good)
    try:
        fixed = s.encode('latin-1').decode('utf-8')
        if fixed != s and '\ufffd' not in fixed:
            s = fixed
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    s = unicodedata.normalize('NFC', s)
    return s.strip()

In [ ]:
STRING_COLS = [
    'winner_name', 'loser_name', 'tourney_name',
    'winner_ioc', 'loser_ioc',
    'winner_hand', 'loser_hand', 'surface',
]

before_names = df['winner_name'].copy()
for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].map(normalize_text)

changed_mask = before_names.fillna('') != df['winner_name'].fillna('')
n_changed = int(changed_mask.sum())
print(f'winner_name values changed by normalization: {n_changed}')
if n_changed > 0:
    sample = pd.DataFrame({
        'old': before_names[changed_mask],
        'new': df.loc[changed_mask, 'winner_name'],
    }).drop_duplicates('old').head(10)
    print('\nSample of renormalized winner_name values:')
    print(sample.to_string(index=False))

In [ ]:
def canonical_hand(v):
    if not isinstance(v, str):
        return 'U'
    t = v.strip()
    if t in ('R', 'Right-Handed', 'Right'):
        return 'R'
    if t in ('L', 'Left-Handed', 'Left'):
        return 'L'
    return 'U'

df['winner_hand'] = df['winner_hand'].map(canonical_hand)
df['loser_hand']  = df['loser_hand'].map(canonical_hand)
print('winner_hand values:', df['winner_hand'].value_counts(dropna=False).to_dict())
print('loser_hand values: ', df['loser_hand'].value_counts(dropna=False).to_dict())
print(f'\u2713 Section 3 complete \u2014 {n_changed} name values normalized')

## Section 4 — Filter invalid matches

Walkovers, retirements, and abandoned matches carry incomplete / misleading serve statistics. Davis Cup ties are dropped because ~63% of their serve-stat rows are surface-median imputations (home venues, non-standard surfaces) — they are not representative of tour play. Drop all of these before imputation so they do not contaminate surface-level medians.

In [ ]:
start_n = len(df)
score_str = df['score'].fillna('').astype(str)

mask_wo = score_str.str.strip().str.upper().str.startswith('W/O') | score_str.str.strip().eq('W/O')
n_wo = int(mask_wo.sum())
df = df.loc[~mask_wo].copy()
print(f'4a) Walkovers removed: {n_wo}')

score_str = df['score'].fillna('').astype(str)
mask_ret = score_str.str.contains(r'RET', case=False, regex=True, na=False)
n_ret = int(mask_ret.sum())
df = df.loc[~mask_ret].copy()
print(f'4b) Retirements removed: {n_ret}')

score_str = df['score'].fillna('').astype(str)
mask_abn = score_str.str.contains(r'ABN|DEF|In Progress', case=False, regex=True, na=False)
n_abn = int(mask_abn.sum())
df = df.loc[~mask_abn].copy()
print(f'4c) Abandoned / incomplete removed: {n_abn}')

mask_dc = df['tourney_level'] == 'D'
n_dc = int(mask_dc.sum())
df = df.loc[~mask_dc].copy()
print(f'4d) Davis Cup ties removed: {n_dc}')

removed = start_n - len(df)
print(f'\n4e) Total rows removed: {removed}  |  rows remaining: {len(df)}')
print(f'\u2713 Section 4 complete \u2014 {removed} invalid/excluded matches dropped')

## Section 5 — Surface cleaning

Collapse case / whitespace variations to the canonical `Hard / Clay / Grass`. Carpet is remapped to Hard: the surface was phased out of the ATP tour and replaced by indoor hard courts — the speed characteristics are comparable and 65 rows is too small a sample for a separate surface category. Any remaining null surfaces default to Hard.

In [ ]:
print('Unique surface values BEFORE:', sorted(df['surface'].dropna().unique().tolist()))

SURFACE_MAP = {
    'hard': 'Hard', 'h': 'Hard',
    'clay': 'Clay', 'c': 'Clay',
    'grass': 'Grass', 'g': 'Grass',
    'carpet': 'Hard', 'cpt': 'Hard',  # Carpet phased out; remapped to Hard
}

def canon_surface(v):
    if not isinstance(v, str):
        return v
    key = v.strip().lower()
    return SURFACE_MAP.get(key, v.strip().title())

df['surface'] = df['surface'].map(canon_surface)
print('Unique surface values AFTER: ', sorted(df['surface'].dropna().unique().tolist()))

In [ ]:
missing_mask = df['surface'].isna()
n_missing = int(missing_mask.sum())
print(f'Rows with missing surface: {n_missing}')
if n_missing:
    df.loc[missing_mask, 'surface'] = 'Hard'
    print(f'Defaulted {n_missing} null surfaces to Hard')

assert df['surface'].isna().sum() == 0, 'surface should have no nulls'
print(f'\u2713 Section 5 complete \u2014 surface null rate = 0%')

## Section 6 — Enrich from players database

`winner_hand`/`loser_hand` and `winner_ht`/`loser_ht` have many missing values in the match files. Fill them from the enriched player database by matching `name_first + ' ' + name_last`.

In [ ]:
p = df_players.copy()
for c in ('name_first', 'name_last', 'hand', 'ioc'):
    if c in p.columns:
        p[c] = p[c].map(normalize_text)

p['full_name'] = (p['name_first'].fillna('') + ' ' + p['name_last'].fillna('')).str.strip()
p['hand_std'] = p['hand'].map(canonical_hand)

player_lookup = (
    p.dropna(subset=['full_name'])
     .drop_duplicates('full_name', keep='first')
     .set_index('full_name')[['hand_std', 'height', 'dob', 'ioc']]
     .to_dict('index')
)
print(f'Player lookup size: {len(player_lookup)}')

In [ ]:
def lookup_field(name, field):
    rec = player_lookup.get(name)
    if rec is None:
        return np.nan
    v = rec.get(field)
    if pd.isna(v):
        return np.nan
    return v

fill_counts = {}

for side in ('winner', 'loser'):
    col = f'{side}_hand'
    mask = df[col].eq('U') | df[col].isna()
    looked = df.loc[mask, f'{side}_name'].map(lambda n: lookup_field(n, 'hand_std'))
    looked = looked.where(looked.isin(['R', 'L']))
    filled = looked.notna().sum()
    df.loc[mask, col] = looked.fillna(df.loc[mask, col])
    fill_counts[col] = int(filled)

for side in ('winner', 'loser'):
    col = f'{side}_ht'
    mask = df[col].isna()
    looked = df.loc[mask, f'{side}_name'].map(lambda n: lookup_field(n, 'height'))
    filled = looked.notna().sum()
    df.loc[mask, col] = looked
    fill_counts[col] = int(filled)

for k, v in fill_counts.items():
    print(f'  filled {v:>6} values in {k}')
print(f'\u2713 Section 6 complete \u2014 {sum(fill_counts.values())} total values enriched')

## Section 7 — Missing stats imputation

After dropping retirements/walkovers, the remaining missing serve stats are data-collection gaps. Impute with the **surface-specific median** so Wimbledon aces stay high, Roland Garros stay low, etc.

In [ ]:
STAT_COLS = [
    'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
    'w_SvGms', 'w_bpSaved', 'w_bpFaced',
    'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon',
    'l_SvGms', 'l_bpSaved', 'l_bpFaced',
]
for c in STAT_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')

medians = df.groupby('surface')[STAT_COLS].median()
print('Surface-level medians:')
print(medians.round(2))

In [ ]:
imputed_counts = {c: 0 for c in STAT_COLS}
for col in STAT_COLS:
    for surface in medians.index:
        mask = (df['surface'] == surface) & df[col].isna()
        n = int(mask.sum())
        if n:
            df.loc[mask, col] = medians.loc[surface, col]
            imputed_counts[col] += n

for c, n in imputed_counts.items():
    print(f'  imputed {n:>6} values in {c}')
print(f'\u2713 Section 7 complete \u2014 {sum(imputed_counts.values())} total cells imputed')

## Section 8 — Derived columns

Create the proper `match_date`, serve-percentage ratios, and break-point conversion rates that the next notebook expects.

In [ ]:
df['match_date'] = pd.to_datetime(df['tourney_date'], format='%Y%m%d', errors='coerce')
print(f"match_date range: {df['match_date'].min()}  \u2192  {df['match_date'].max()}")

df['w_1stServe_pct'] = df['w_1stIn']  / df['w_svpt']
df['w_1stWon_pct']   = df['w_1stWon'] / df['w_1stIn']
df['w_2ndWon_pct']   = df['w_2ndWon'] / (df['w_svpt'] - df['w_1stIn'])
df['l_1stServe_pct'] = df['l_1stIn']  / df['l_svpt']
df['l_1stWon_pct']   = df['l_1stWon'] / df['l_1stIn']
df['l_2ndWon_pct']   = df['l_2ndWon'] / (df['l_svpt'] - df['l_1stIn'])

pct_cols = ['w_1stServe_pct', 'w_1stWon_pct', 'w_2ndWon_pct',
            'l_1stServe_pct', 'l_1stWon_pct', 'l_2ndWon_pct']
for c in pct_cols:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan).clip(0, 1)

df['w_bp_conversion'] = (df['l_bpFaced'] - df['l_bpSaved']) / df['l_bpFaced'].clip(lower=1)
df['l_bp_conversion'] = (df['w_bpFaced'] - df['w_bpSaved']) / df['w_bpFaced'].clip(lower=1)
df['w_bp_conversion'] = df['w_bp_conversion'].clip(0, 1)
df['l_bp_conversion'] = df['l_bp_conversion'].clip(0, 1)

print(f'\u2713 Section 8 complete \u2014 derived 9 new columns')

## Section 9 — Final validation & save

Run assertions, print a summary table, and persist both a Parquet (canonical, for the Databricks step) and a CSV (for human inspection) to the Workspace datasets folder.

In [ ]:
def check(label, ok):
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return ok

print('Validation:')
required = ['surface', 'winner_name', 'loser_name', 'match_date', 'tourney_level', 'round']
check('no nulls in required cols', all(df[c].notna().all() for c in required))
check('surface in {Hard, Clay, Grass}', set(df['surface'].unique()).issubset({'Hard', 'Clay', 'Grass'}))
check('no Davis Cup rows', (df['tourney_level'] != 'D').all())
check('winner_hand in {R, L, U}', set(df['winner_hand'].dropna().unique()).issubset({'R', 'L', 'U'}))
check('loser_hand in {R, L, U}',  set(df['loser_hand'].dropna().unique()).issubset({'R', 'L', 'U'}))
check('no W/O in score', not df['score'].fillna('').astype(str).str.contains(r'W/O', case=False).any())
check('no RET in score', not df['score'].fillna('').astype(str).str.contains(r'RET', case=False).any())
pct_cols = ['w_1stServe_pct','w_1stWon_pct','w_2ndWon_pct','l_1stServe_pct','l_1stWon_pct','l_2ndWon_pct']
check('pct cols in [0,1] or NaN', all(df[c].dropna().between(0, 1).all() for c in pct_cols))
check('match_date within 2016\u20132025', df['match_date'].min().year >= 2016 and df['match_date'].max().year <= 2025)
check('row count > 20,000', len(df) > 20_000)

In [ ]:
stat_null_rate = df[STAT_COLS].isna().mean().mean() * 100
unique_players = pd.unique(pd.concat([df['winner_name'], df['loser_name']], ignore_index=True).dropna()).size
surfaces_str = '/'.join(sorted(df['surface'].unique()))

summary_lines = [
    '\u250c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2510',
    '\u2502 ATP Clean Dataset \u2014 Summary         \u2502',
    '\u251c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u252c\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2524',
    f'\u2502 Total matches     \u2502 {len(df):<15} \u2502',
    f"\u2502 Years covered     \u2502 {int(df['year'].min())}\u2013{int(df['year'].max()):<10} \u2502",
    f'\u2502 Surfaces          \u2502 {surfaces_str:<15} \u2502',
    f'\u2502 Unique players    \u2502 {unique_players:<15} \u2502',
    f'\u2502 Columns           \u2502 {df.shape[1]:<15} \u2502',
    f'\u2502 Null rate (stats) \u2502 {stat_null_rate:<14.2f}% \u2502',
    '\u2514\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2534\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2518',
]
print('\n'.join(summary_lines))

In [ ]:
parquet_path = os.path.join(OUTPUT_DIR, 'atp_matches_clean_final.parquet')
csv_path     = os.path.join(OUTPUT_DIR, 'atp_matches_clean_final.csv')

df.to_parquet(parquet_path, engine='pyarrow', index=False)
df.to_csv(csv_path, index=False)

def human(size):
    for unit in ('B', 'KB', 'MB', 'GB'):
        if size < 1024:
            return f'{size:.2f} {unit}'
        size /= 1024
    return f'{size:.2f} TB'

print(f'Parquet: {parquet_path}  ({human(os.path.getsize(parquet_path))})')
print(f'CSV:     {csv_path}  ({human(os.path.getsize(csv_path))})')
print(f'\u2713 Section 9 complete \u2014 canonical dataset written')